# Check GV Tools 0.50.1

This notebook demonstrates loading and plotting every supported instrument family. Set each input to one absolute filename, one absolute directory, or a list of compatible absolute paths. GV Tools never chooses data-output locations; every figure filename below is explicitly supplied by the user.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import gv_tools

print('gv_tools', gv_tools.__version__)
assert gv_tools.__version__ == '0.50.1'

## User configuration
The defaults below are real sample paths used previously on this workstation. PIERS instruments default to consolidated daily products. Every output filename is explicit and is written under WORKSPACE / 'Output'.

In [ ]:
RUN_EXAMPLES = True

WORKSPACE = Path('/Users/dwolff/Desktop/Work/GV_Tools')
PACKAGE_DIRECTORY = WORKSPACE / 'gv_tools'
OUTPUT_DIRECTORY = WORKSPACE / 'Output'

PIERS_YEAR = 2026
PIERS_MONTH = 10
PIERS_DAY = 11
PIERS_ID = 'PIERS0045'
PIERS_DATE = f'{PIERS_YEAR:04d}{PIERS_MONTH:02d}{PIERS_DAY:02d}'
PIERS_DATE_TEXT = f'{PIERS_YEAR:04d}-{PIERS_MONTH:02d}-{PIERS_DAY:02d}'

PIERS_DAILY_ROOT = Path('/Volumes/TBW/distro/PIERS_Gauges_Daily')
PIERS_DAY_DIRECTORY = (
    PIERS_DAILY_ROOT
    / PIERS_ID
    / f'{PIERS_YEAR:04d}'
    / f'{PIERS_MONTH:02d}'
    / f'{PIERS_DAY:02d}'
)

RM_YOUNG_AIO_PATH = PIERS_DAY_DIRECTORY / f'{PIERS_ID}_WX_{PIERS_DATE}_daily.csv'
PIERS_GAUGE_PATH = PIERS_DAY_DIRECTORY / f'{PIERS_ID}_TB_{PIERS_DATE}_daily.csv'
PIERS_PARSIVEL_PATH = PIERS_DAY_DIRECTORY / f'{PIERS_ID}_Parsivel_{PIERS_DATE}_daily.zip'
LUFFT_WS800_PATH = Path(
    '/Volumes/TBW/distro/PIERS_Gauges_Daily/PIERS0042/2026/10/11/'
    'PIERS0042_WS800_20261011_daily.csv'
)
METONE_AIO_PATH = Path('/Volumes/TBW/distro/aio/AIO_GAIL_202501.dat')
MRR_PATH = Path('/Volumes/36TB/MRR/OU/202402/20240228/20240228_032000.nc')
RADAR_PATH = Path(
    '/Volumes/36TB/Radars/KCRP/cf/08-2024/240821/'
    'KCRP_2024_0821_081505.cf.gz'
)
DVD_PATH = Path('/Volumes/TBW/distro/2dvd/2dvd_sn37/ascii/V23022.drops.zip')

RM_YOUNG_OUTPUT = OUTPUT_DIRECTORY / f'{PIERS_ID}-RMYoung_{PIERS_DATE_TEXT}_quicklook.png'
WS800_OUTPUT = OUTPUT_DIRECTORY / f'PIERS0042-WS800_{PIERS_DATE_TEXT}_quicklook.png'
METONE_OUTPUT = OUTPUT_DIRECTORY / 'GAIL-MetOne-AIO_2025-01-01_quicklook.png'
PARSIVEL_OUTPUT = OUTPUT_DIRECTORY / f'{PIERS_ID}-Parsivel_{PIERS_DATE_TEXT}_quicklook.png'
GAUGE_OUTPUT = OUTPUT_DIRECTORY / f'{PIERS_ID}-TippingBucket_{PIERS_DATE_TEXT}_timeseries.png'
MRR_OUTPUT = OUTPUT_DIRECTORY / 'MRRPro86_2024-02-28_032000_time-height.png'
RADAR_PPI_OUTPUT = OUTPUT_DIRECTORY / 'KCRP_2024-08-21_081505_ppi.png'
RADAR_RHI_OUTPUT = OUTPUT_DIRECTORY / 'KCRP_2024-08-21_081505_rhi.png'
DVD_OUTPUT = OUTPUT_DIRECTORY / '2DVD-sn37_2023-01-22_drop-velocity.png'

INPUTS = {
    'rm_young_aio': RM_YOUNG_AIO_PATH,
    'lufft_ws800': LUFFT_WS800_PATH,
    'metone_aio': METONE_AIO_PATH,
    'parsivel': PIERS_PARSIVEL_PATH,
    'gauge': PIERS_GAUGE_PATH,
    'mrr': MRR_PATH,
    'radar': RADAR_PATH,
    '2dvd': DVD_PATH,
}

if RUN_EXAMPLES:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)


## R.M. Young All-in-One weather station

In [ ]:
if RUN_EXAMPLES:
    rm_young = gv_tools.io.read_aio(INPUTS['rm_young_aio'], XARRAY=True)
    fig = gv_tools.graph.plot_aio_quicklook(rm_young, savefig=RM_YOUNG_OUTPUT)
    display(rm_young)


## OTT HydroMet Lufft WS800

In [ ]:
if RUN_EXAMPLES:
    ws800 = gv_tools.io.read_ws800_full(INPUTS['lufft_ws800'], XARRAY=True)
    fig = gv_tools.graph.plot_ws800_full_quicklook(ws800, savefig=WS800_OUTPUT)
    plt.close(fig)
    display(ws800)


## Met One All-in-One weather station

In [ ]:
if RUN_EXAMPLES:
    metone = gv_tools.io.read_aio(INPUTS['metone_aio'], XARRAY=True)
    fig = gv_tools.graph.plot_aio_quicklook(metone, savefig=METONE_OUTPUT)
    display(metone)


## OTT Parsivel and Parsivel2 disdrometers

In [ ]:
if RUN_EXAMPLES:
    parsivel = gv_tools.io.read_parsivel(INPUTS['parsivel'])
    fig = gv_tools.graph.plot_parsivel_quicklook(parsivel, savefig=PARSIVEL_OUTPUT)
    display(parsivel)


## PIERS tipping-bucket rain gauge
Gauge measurements are plotted against packet timestamps with the explicitly named general field plotter.

In [ ]:
if RUN_EXAMPLES:
    gauge = gv_tools.io.read_gauge(INPUTS['gauge'], measurement='TB_A_tips')
    fig, ax = gv_tools.graph.plot_instrument_timeseries(
        gauge, fields='value', time='timestamp', output=GAUGE_OUTPUT,
        title='Tipping-bucket gauge measurements', ylabel='Recorded value',
    )
    display(gauge)


## Micro Rain Radar MRR2 or MRRPro
Select fields that exist in your product; common post-processed products include `Ze`, `VEL`, and `RR`.

In [ ]:
if RUN_EXAMPLES:
    mrr = gv_tools.io.read_mrr(INPUTS['mrr'])
    mrr_fields = ['Ze', 'VEL', 'WIDTH', 'RR']
    fig = gv_tools.graph.plot_mrr_time_height_quicklook(
        mrr, mrr_fields, savefig=MRR_OUTPUT
    )
    display(mrr)


## Weather radar
`read_radar` detects NEXRAD Level II, SIGMET/IRIS, Universal Format, and CF/Radial. Use the PPI or RHI plotter appropriate for the scan.

In [ ]:
if RUN_EXAMPLES:
    radar = gv_tools.io.read_radar(INPUTS['radar'])
    radar_field = next(iter(radar.fields))
    scan_type = str(getattr(radar, 'scan_type', '')).lower()
    if scan_type == 'rhi':
        fig = gv_tools.graph.plot_radar_rhi_quicklook(
            radar, field=radar_field, savefig=RADAR_RHI_OUTPUT
        )
    else:
        fig = gv_tools.graph.plot_radar_ppi_quicklook(
            radar, field=radar_field, add_map_features=False,
            savefig=RADAR_PPI_OUTPUT
        )
    print(gv_tools.io.inspect_radar(INPUTS['radar']))


## Two-Dimensional Video Disdrometer (2DVD)
Each row is an individual detected drop. The example plots measured drop velocity against detection time; it performs no aggregation or scientific analysis.

In [ ]:
if RUN_EXAMPLES:
    dvd = gv_tools.io.read_2dvd(INPUTS['2dvd'])
    fig, ax = gv_tools.graph.plot_instrument_timeseries(
        dvd, fields='measured_velocity_m_s', time='time', output=DVD_OUTPUT,
        title='2DVD detected-drop velocity', ylabel='Velocity (m s-1)',
    )
    display(dvd)


## User-owned data output
GV Tools deliberately does not select or create data-product filenames. After reviewing an object, use pandas or xarray directly, for example: `gauge.to_csv('/your/path/gauge.csv')` or `mrr.to_netcdf('/your/path/mrr.nc')`.